# Day 3 (Part B & C) — Model Preparation + Baseline Model
Real Estate Price Prediction (Gujrat, Gujranwala, Sialkot — Zameen.com)

Is notebook mein hum:
1. Categorical variables encode karte hain
2. Train-test split karte hain
3. Linear Regression baseline model train karte hain
4. Evaluation metrics (RMSE, MAE, R²) calculate karte hain

## 1. Data load karein

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv("../data/clean_data.csv")
df.head()

,Title,Price_Raw,Price_PKR,Size_Raw,Size_Marla,Listing_URL,City,Property_Type,Bedrooms,Bathrooms,Location,Price_per_Marla
0,River Garden 5 Marla House for sale,2.25 Crore,22500000.0,5.0 Marla,5.0,https://www.zameen.com/Property/gt_road_river_...,Gujrat,House,3.0,4.0,"GT Road, Gujrat, Punjab",4.500000e+06
1,5 Marla Newly Constructed Furnished House For ...,2.75 Crore,27500000.0,5.0 Marla,5.0,https://www.zameen.com/Property/gujrat_shadman...,Gujrat,House,6.0,5.0,"Shadman Colony, Gujrat, Punjab",5.500000e+06
2,Prime Commercial-Location House for Sale Near ...,6.5 Crore,65000000.0,14.0 Marla,14.0,https://www.zameen.com/Property/gujrat_bara_da...,Gujrat,House,NaN,NaN,NaN,4.642857e+06
3,10 Marla Luxury House,4.3 Crore,43000000.0,10.0 Marla,10.0,https://www.zameen.com/Property/gt_road_river_...,Gujrat,House,NaN,NaN,"GT Road, Gujrat, Punjab",4.300000e+06
4,5 Marla Luxury Designer House,2.85 Crore,28500000.0,5.0 Marla,5.0,https://www.zameen.com/Property/gt_road_river_...,Gujrat,House,4.0,6.0,"GT Road, Gujrat, Punjab",5.700000e+06


## 2. Categorical variables encode karein
`City` aur `Property_Type` ko One-Hot Encoding se numeric columns mein convert kar rahe hain
(`pd.get_dummies`), taake model inhe samajh sake.

In [2]:
model_df = pd.get_dummies(df, columns=["City", "Property_Type"], drop_first=True)
model_df.head()

,Title,Price_Raw,Price_PKR,Size_Raw,Size_Marla,Listing_URL,Bedrooms,Bathrooms,Location,Price_per_Marla,City_Gujrat,City_Sialkot
0,River Garden 5 Marla House for sale,2.25 Crore,22500000.0,5.0 Marla,5.0,https://www.zameen.com/Property/gt_road_river_...,3.0,4.0,"GT Road, Gujrat, Punjab",4.500000e+06,True,False
1,5 Marla Newly Constructed Furnished House For ...,2.75 Crore,27500000.0,5.0 Marla,5.0,https://www.zameen.com/Property/gujrat_shadman...,6.0,5.0,"Shadman Colony, Gujrat, Punjab",5.500000e+06,True,False
2,Prime Commercial-Location House for Sale Near ...,6.5 Crore,65000000.0,14.0 Marla,14.0,https://www.zameen.com/Property/gujrat_bara_da...,NaN,NaN,NaN,4.642857e+06,True,False
3,10 Marla Luxury House,4.3 Crore,43000000.0,10.0 Marla,10.0,https://www.zameen.com/Property/gt_road_river_...,NaN,NaN,"GT Road, Gujrat, Punjab",4.300000e+06,True,False
4,5 Marla Luxury Designer House,2.85 Crore,28500000.0,5.0 Marla,5.0,https://www.zameen.com/Property/gt_road_river_...,4.0,6.0,"GT Road, Gujrat, Punjab",5.700000e+06,True,False


## 3. Features (X) aur Target (y) alag karein
`Price_per_Marla` ko features se hata rahe hain kyunke ye seedha `Price_PKR` se derive hui hai
(agar isay feature rakhein to model "cheat" kar lega — ye data leakage kehlata hai).

In [3]:
feature_cols = [c for c in model_df.columns
                if c not in ["Price_PKR", "Price_per_Marla", "Title", "Price_Raw",
                             "Size_Raw", "Listing_URL"]]

X = model_df[feature_cols]
y = model_df["Price_PKR"]

print("Features used:", feature_cols)
print("X shape:", X.shape, "| y shape:", y.shape)

Features used: ['Size_Marla', 'Bedrooms', 'Bathrooms', 'Location', 'City_Gujrat', 'City_Sialkot']
X shape: (1317, 6) | y shape: (1317,)


## 4. Train-test split (80-20)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])

Train size: 1053 | Test size: 264


## 5. Feature scaling
Linear Regression ke liye zaroori nahi hota, lekin achi practice hai — aur agar baad mein
Regularization (Ridge/Lasso) try karein to zaroorat pare gi.

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train.isnull().sum()
X_train.dtypes

ValueError: could not convert string to float: 'Citi Housing Society, Gujranwala, Punjab'

## 6. Baseline model: Linear Regression

In [7]:
print(feature_cols)

['Size_Marla', 'Bedrooms', 'Bathrooms', 'Location', 'City_Gujrat', 'City_Sialkot']


In [6]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

## 7. Evaluation metrics

In [7]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:,.0f} PKR")
print(f"MAE:  {mae:,.0f} PKR")
print(f"R2 Score: {r2:.3f}")

RMSE: 5,776,351 PKR
MAE:  4,243,069 PKR
R2 Score: 0.827


In [8]:
coef_df = pd.DataFrame({ "Feature": feature_cols, "Effect_per_unit_PKR": model.coef_ / scaler.scale_ }).sort_values("Effect_per_unit_PKR", key=abs, ascending=False) 
print(f"Base amount (intercept): {model.intercept_:,.0f} PKR\n") 
coef_df

Base amount (intercept): 31,303,039 PKR



,Feature,Effect_per_unit_PKR
0,Size_Marla,4.338138e+06
1,City_Gujrat,1.729842e+06
2,City_Sialkot,1.402982e+06


In [9]:
df["Property_Type"].value_counts()

Property_Type
House    1317
Name: count, dtype: int64

## Notes
- **RMSE/MAE** PKR mein hain — jitna kam, utna acha (average error kitna hai rupees mein).
- **R² Score** 0 se 1 ke darmiyan hota hai — 1 ka matlab perfect predictions, 0 ka matlab model
  koi pattern nahi seekh raha. 0.6-0.8 range ek decent baseline ke liye normal hai.
- Ye numbers Day 4 mein Random Forest/XGBoost ke sath compare karenge — dekhenge kitna improvement milta hai.